<a href="https://colab.research.google.com/github/JairusTheAnalyst/JairusTheAnalyst/blob/main/Project_Analyzing_Syntax_and_Semantics_in_Text_Using_NLP_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Project: Analyzing Syntax and Semantics in Text Using NLP Tools**

In [1]:
# Install necessary libraries
!pip install spacy
!pip install sentence-transformers

# Download English model for spaCy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


**Step 2: Import Libraries**

In [2]:
import spacy
from sentence_transformers import SentenceTransformer, util

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

# Load SBERT model for semantic similarity
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Step 3: Define Sample Text**

In [3]:
text = """
The quick brown fox jumps over the lazy dog.
Apple is looking to buy a startup in the UK.
I love NLP, but I hate waiting in long queues.
"""

**Step 4: Syntactic Analysis (POS & Dependency Parsing)**

In [4]:
# Process text with spaCy
doc = nlp(text)

print("=== Part-of-Speech (POS) Tagging ===")
for token in doc:
    print(f"{token.text:10} {token.pos_:10} {token.tag_:10} {token.dep_:10} {token.head.text}")

print("\n=== Dependency Parsing ===")
for sent in doc.sents:
    for token in sent:
        print(f"{token.text:10} --> {token.dep_:10} --> {token.head.text}")

=== Part-of-Speech (POS) Tagging ===

          SPACE      _SP        dep        The
The        DET        DT         det        fox
quick      ADJ        JJ         amod       fox
brown      ADJ        JJ         amod       fox
fox        NOUN       NN         nsubj      jumps
jumps      VERB       VBZ        ROOT       jumps
over       ADP        IN         prep       jumps
the        DET        DT         det        dog
lazy       ADJ        JJ         amod       dog
dog        NOUN       NN         pobj       over
.          PUNCT      .          punct      jumps

          SPACE      _SP        dep        .
Apple      PROPN      NNP        nsubj      looking
is         AUX        VBZ        aux        looking
looking    VERB       VBG        ROOT       looking
to         PART       TO         aux        buy
buy        VERB       VB         xcomp      looking
a          DET        DT         det        startup
startup    NOUN       NN         dobj       buy
in         ADP        IN

**Step 5: Semantic Analysis Using SBERT**

In [5]:
# Split text into sentences
sentences = [sent.text.strip() for sent in doc.sents]

# Encode sentences into embeddings
embeddings = sbert_model.encode(sentences, convert_to_tensor=True)

# Compute semantic similarity between all pairs
cosine_scores = util.cos_sim(embeddings, embeddings)

print("=== Semantic Similarity Matrix ===")
for i in range(len(sentences)):
    for j in range(len(sentences)):
        print(f"Similarity between '{sentences[i]}' and '{sentences[j]}': {cosine_scores[i][j]:.2f}")

=== Semantic Similarity Matrix ===
Similarity between 'The quick brown fox jumps over the lazy dog.' and 'The quick brown fox jumps over the lazy dog.': 1.00
Similarity between 'The quick brown fox jumps over the lazy dog.' and 'Apple is looking to buy a startup in the UK.': 0.08
Similarity between 'The quick brown fox jumps over the lazy dog.' and 'I love NLP, but I hate waiting in long queues.': 0.11
Similarity between 'Apple is looking to buy a startup in the UK.' and 'The quick brown fox jumps over the lazy dog.': 0.08
Similarity between 'Apple is looking to buy a startup in the UK.' and 'Apple is looking to buy a startup in the UK.': 1.00
Similarity between 'Apple is looking to buy a startup in the UK.' and 'I love NLP, but I hate waiting in long queues.': -0.02
Similarity between 'I love NLP, but I hate waiting in long queues.' and 'The quick brown fox jumps over the lazy dog.': 0.11
Similarity between 'I love NLP, but I hate waiting in long queues.' and 'Apple is looking to buy 

**Step 7: Optional Visualization (spaCy)**

In [6]:
from spacy import displacy

# Visualize dependency tree for first sentence
displacy.render(list(doc.sents)[0], style='dep', jupyter=True, options={'distance': 100})

1. Diagonal values (self-similarity = 1.00)

  •	Each sentence compared to itself has a similarity of 1.00, which is expected.

  •	This confirms that the embeddings are correctly representing the sentence meaning.

2. Low similarity between unrelated sentences

  •	'The quick brown fox jumps over the lazy dog.' vs 'Apple is looking to buy a startup in the UK.' → 0.08

  •	'The quick brown fox jumps over the lazy dog.' vs 'I love NLP, but I hate waiting in long queues.' → 0.11

  •	'Apple is looking to buy a startup in the UK.' vs 'I love NLP, but I hate waiting in long queues.' → -0.02

**Interpretation:**
•	These sentences are semantically unrelated, so SBERT correctly assigns very low similarity.
•	Negative values (like -0.02) indicate opposite or dissimilar semantic directions in embedding space.

3. Patterns in similarity

•	Similarity values are closer to 0 for unrelated topics, showing that the model distinguishes topics and meaning rather than just syntactic structure.

•	For example:

o	'The quick brown fox jumps over the lazy dog.' and 'I love NLP…' share slightly positive similarity (0.11).

This might reflect general language overlap (verbs, structure), but overall, the sentences are unrelated in meaning.

4. Overall takeaway

•	Syntactic structure alone does not determine semantic similarity. For instance, 'The quick brown fox…' and 'Apple is looking to buy…' both have noun-verb structures, but embeddings capture contextual meaning, not just grammar.

•	SBERT embeddings effectively separate sentences by meaning, highlighting semantic coherence in text analysis.

**Summary Table of Insights:**

Sentence Pair	Similarity	Interpretation
Sentence vs itself	1.00

Perfect self-similarity
'fox' vs 'Apple startup'	0.08
Different topics → low similarity
'fox' vs 'NLP queues'	0.11
Slight language overlap, low meaning similarity
'Apple startup' vs 'NLP queues'	-0.02

Semantically opposite → negative similarity

Conclusion:

•	Semantic similarity embeddings capture meaning beyond surface words or syntax.

•	This allows your NLP system to detect coherence, topic similarity, or semantic relationships, which is key for tasks like question answering, summarization, or text clustering.

